<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ALIGNN_Sc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Notebook that uses JARVIS-DFT data to train an ALIGNN model for prediction of Tc for BCS superconductors:

Refs:

https://www.nature.com/articles/s41524-022-00933-1

https://pubs.acs.org/doi/full/10.1021/acs.nanolett.2c04420

https://pubs.acs.org/doi/10.1021/acs.jpclett.3c01260


In [ ]:
%%time
!pip install -q dgl==1.0.1+cu117 -f https://data.dgl.ai/wheels/cu117/repo.html
!pip install -q alignn

# For this example, we only use 3 epochs and a batch size of 2. For production-level calculations this will be increased.

In [ ]:
import json

data = {
    "version": "112bbedebdaecf59fb18e11c929080fb2f358246",
    "dataset": "user_data",
    "target": "target",
    "atom_features": "cgcnn",
    "neighbor_strategy": "k-nearest",
    "id_tag": "jid",
    "random_seed": 123,
    "classification_threshold": None,
    "n_val": None,
    "n_test": None,
    "n_train": None,
    "train_ratio": 0.8,
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "target_multiplication_factor": None,
    "epochs": 3,
    "batch_size": 2,
    "weight_decay": 1e-05,
    "learning_rate": 0.001,
    "filename": "sample",
    "warmup_steps": 2000,
    "criterion": "mse",
    "optimizer": "adamw",
    "scheduler": "onecycle",
    "pin_memory": False,
    "save_dataloader": False,
    "write_checkpoint": True,
    "write_predictions": True,
    "store_outputs": True,
    "progress": True,
    "log_tensorboard": False,
    "standard_scalar_and_pca": False,
    "use_canonize": True,
    "num_workers": 0,
    "cutoff": 8.0,
    "max_neighbors": 12,
    "keep_data_order": False,
    "model": {
        "name": "alignn",
        "alignn_layers": 4,
        "gcn_layers": 4,
        "atom_input_features": 92,
        "edge_input_features": 80,
        "triplet_input_features": 40,
        "embedding_features": 64,
        "hidden_features": 256,
        "output_features": 1,
        "link": "identity",
        "zero_inflated": False,
        "classification": False
    }
}

file_path = "config.json"

with open(file_path, "w") as json_file:
    json.dump(data, json_file)

# Generate the dataset from JARVIS-DFT database of 1k superconducting materials (electron-phonon coupling calculations): https://www.nature.com/articles/s41524-022-00933-1

In [ ]:
"""Module to generate example dataset."""
from jarvis.db.figshare import data as jdata
from jarvis.core.atoms import Atoms
outdir = "./"
dft_3d = jdata("dft_3d")
prop = "Tc_supercon"  # "optb88vdw_bandgap"
# max_samples = 50
f = open("id_prop.csv", "w")
count = 0
for i in dft_3d:
    atoms = Atoms.from_dict(i["atoms"])
    jid = i["jid"]
    poscar_name = "POSCAR-" + jid + ".vasp"
    target = i[prop]
    if target != "na":
        atoms.write_poscar(poscar_name)
        f.write("%s,%6f\n" % (poscar_name, target))
        # count += 1
        # if count == max_samples:
        #    break
f.close()

# Train ALIGNN model on DFT dataset

In [ ]:
%%time
!train_folder.py --root_dir "./" --config "config.json" --output_dir=temp

# The results are contained in the temp folder. The checkpoint_3.pt file is the trained model for the 3rd and final epoch.

In [ ]:
!ls temp

# Plot the predicted results vs. target (DFT) results for the test set and training set.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
data = pd.read_csv("temp/prediction_results_test_set.csv", skiprows=1)

x = data.iloc[:, 1]  # Second column
y = data.iloc[:, 2]  # Third column
line_values = np.linspace(min(x.min(), y.min()), max(x.max(), y.max()), 100)
plt.plot(line_values, line_values, label="x=y Reference", color="r", linestyle="--")
plt.scatter(x, y)
plt.xlabel("Target DFT Tc (K)")
plt.ylabel("Predicted ALIGNN (Tc)")
plt.title("Prediction Results for Test Set")
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
data = pd.read_csv("temp/prediction_results_train_set.csv", skiprows=1)

x = data.iloc[:, 0]  # Second column
y = data.iloc[:, 1]  # Third column
line_values = np.linspace(min(x.min(), y.min()), max(x.max(), y.max()), 100)
plt.plot(line_values, line_values, label="x=y Reference", color="r", linestyle="--")
plt.scatter(x, y)
plt.xlabel("Target DFT Tc (K)")
plt.ylabel("Predicted ALIGNN (Tc)")
plt.title("Prediction Results for Training Set")
plt.show()

# Load our trained ALIGNN model for epoch 3  

In [ ]:
from alignn.models.alignn import ALIGNN, ALIGNNConfig
import torch
output_features =  1
filename = 'temp/checkpoint_3.pt'
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
model = ALIGNN(ALIGNNConfig(name="alignn", output_features=output_features))
model.load_state_dict(torch.load(filename, map_location=device)["model"])
model.eval()

# In this section, we screen for new potential superconductors outside of the training set. Since hydrides have attracted recent attention, we can screen for potential superconductors that contain H.

In [ ]:
from jarvis.db.figshare import data

dft_3d = data('dft_3d')

In [ ]:
df=pd.DataFrame(dft_3d)

In [ ]:
print (df.columns, len(df.columns))

In [ ]:
df1 = df[['optb88vdw_bandgap','magmom_outcar', 'ehull', 'atoms', 'jid']].replace('na',np.nan).dropna()

In [ ]:
from jarvis.core.atoms import Atoms
def has_element(atoms_dict = {}, my_element = 'H'):
    atoms = Atoms.from_dict(atoms_dict)
    elements = atoms.elements
    if my_element in elements:
        return True
    else:
        return np.nan

df1['el'] = df1['atoms'].apply(lambda x: has_element(atoms_dict = x, my_element = 'H'))
df1['formula'] = df1['atoms'].apply(lambda x: Atoms.from_dict(x).composition.reduced_formula)

In [ ]:
df2 = df1.dropna()

We establish our screening criteria below

In [ ]:
df2[(df2['optb88vdw_bandgap'] <= .1) & (df2['ehull'] <= .2) & (df2['magmom_outcar'] <= 0.05)]

JVASP-157789 is a potential superconductor, let's see what our trained model predicts and what the available ALIGNN pretrained model (300 epochs) predicts

In [ ]:
from jarvis.db.figshare import get_jid_data
dat = get_jid_data(jid="JVASP-157789", dataset="dft_3d")

In [ ]:
atoms = Atoms.from_dict(dat["atoms"])

In [ ]:
atoms.write_poscar('POSCAR-JVASP-157789')

In [ ]:
import sys
from alignn.models.alignn import ALIGNN, ALIGNNConfig
import torch
from jarvis.core.atoms import Atoms
from alignn.graphs import Graph

output_features = 1
filename = 'temp/checkpoint_3.pt'
device = "cpu"

if torch.cuda.is_available():
    device = torch.device("cuda")

model = ALIGNN(ALIGNNConfig(name="alignn", output_features=output_features))
model.load_state_dict(torch.load(filename, map_location=device)["model"])
model.eval()

cutoff = 8.0
model = model.to(device)
max_neighbors = 12

jids = ["POSCAR-JVASP-157789"]

# Save the current standard output (e.g., the console) to a variable
original_stdout = sys.stdout

for i in jids:
    atoms = Atoms.from_poscar(i)
    g, lg = Graph.atom_dgl_multigraph(
        atoms, cutoff=float(cutoff), max_neighbors=max_neighbors
    )
    out_data = (
        model([g.to(device), lg.to(device)])
        .detach()
        .cpu()
        .numpy()
        .flatten()
        .tolist()
    )
    print(i, out_data[0])



In [ ]:
!pretrained.py --model_name jv_supercon_tc_alignn --file_format poscar --file_path POSCAR-JVASP-157789

As an excercise, try to find potential superconductors in the JARVIS database and screen them with the ALIGNN model for Tc